# Пилот для h2 и h6
Запуск на GPU в Kaggle. Подключите пару raw parquet, validation_postprocessed.parquet и submission.csv от прогона 0,341. Для сравнения с XGBoost нужны xgb_predictions.parquet и соответствующий submission.csv из одного прогона XGBoost. Ячейка смешивания работает до подготовки признаков и обучения. Все даты оценки используют старт 10:30. Объём полной float32-матрицы для 48 стартов и ~390 признаков достигает 7 ГиБ до внутренних копий CatBoost. Для полного прогона выбирайте сессию с запасом RAM.

In [5]:
from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
DATA = Path('data')
TRAIN_FILE = 'train_solo_track.parquet'
TEST_FILE = 'test_solo_track.parquet'
TRAIN_PATH, TEST_PATH = (DATA / TRAIN_FILE, DATA / TEST_FILE)
if not (TRAIN_PATH.is_file() and TEST_PATH.is_file()) and Path('/kaggle/input').exists():
    train_matches = list(Path('/kaggle/input').rglob(TRAIN_FILE))
    test_matches = list(Path('/kaggle/input').rglob(TEST_FILE))
    if len(train_matches) != 1 or len(test_matches) != 1:
        raise ValueError(f'Укажите TRAIN_PATH и TEST_PATH вручную: train={train_matches}, test={test_matches}')
    TRAIN_PATH, TEST_PATH = (train_matches[0], test_matches[0])
OUT = Path('artifacts/start_pilot/features')
MODEL_DIR = OUT.parent / 'models'
POSTPROCESS_DIR = OUT.parent / 'postprocessing'
for directory in (OUT, MODEL_DIR, POSTPROCESS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TARGET_SCALE = 100000
MAX_TREES = 4000
EARLY_STOPPING_ROUNDS = 150
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
FEATURE_PIPELINE_VERSION = '1030_windows_v4'
FE_VALID = Path("/kaggle/input/datasets/sptrent/val-fe/validation_postprocessed.parquet")
FE_SUB = Path("/kaggle/input/datasets/sptrent/submission-341/submission_fe.csv")
XGB_VALID = Path("artifacts/xgb_direct_5starts/xgb_predictions.parquet")
XGB_SUB = XGB_VALID.with_name("submission.csv")
KEY = ["cutoff", "route_id", "timestamp", "horizon"]
PILOT = (2, 6)

In [6]:
def day_score(frame, pred):
    delta = np.asarray(pred) - frame.y.to_numpy()
    totals = pd.DataFrame({"day":frame.cutoff.dt.normalize(), "y":frame.y,
                           "absolute":np.abs(delta), "signed":delta}).groupby("day").sum()
    return ((totals.absolute + totals.signed.abs()) / totals.y).mean()

def align(left, right, keys):
    merged = left.merge(right, on=keys, how="outer", validate="one_to_one", indicator=True)
    if not merged["_merge"].eq("both").all():
        raise ValueError("Ключи прогнозов не совпадают")
    return merged.drop(columns="_merge")

missing_inputs = [str(path) for path in (TRAIN_PATH, TEST_PATH, FE_VALID, FE_SUB) if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError("Укажите пути к исходным данным и артефактам FE 0,341 в первой ячейке: " + ", ".join(missing_inputs))
contest_keys = pd.read_parquet(TEST_PATH, columns=["id", "route_id", "timestamp"])
fe_future = align(contest_keys, pd.read_csv(FE_SUB, usecols=["id", "y_pred"]), ["id"])
fe_future = fe_future.astype({"y_pred":"float64"})
if not np.isfinite(fe_future.y_pred).all() or fe_future.y_pred.lt(0).any():
    raise ValueError("Недопустимые прогнозы базового сабмита FE")
fe = pd.read_parquet(FE_VALID, columns=KEY + ["period", "y", "y_pred_tuning_calibration"])
fe = fe.rename(columns={"y_pred_tuning_calibration":"fe"}).astype({"fe":"float64"})
fe_route_count = contest_keys.route_id.nunique()
cutoffs = pd.date_range(contest_keys.timestamp.min() - STEP - pd.Timedelta(days=35),
                        periods=35, freq="D")
for period, expected_dates in (("tuning", cutoffs[:21]), ("test", cutoffs[21:])):
    part = fe.loc[fe.period.eq(period)]
    if (len(part) != len(expected_dates)*fe_route_count*8 or part.duplicated(KEY).any() or
        not part.cutoff.isin(expected_dates).all() or not part.route_id.isin(contest_keys.route_id).all() or
        not part.horizon.isin(range(1,9)).all() or not part.timestamp.eq(part.cutoff + part.horizon*STEP).all() or
        not np.isfinite(part[["y", "fe"]]).all().all()):
        raise ValueError(f"Неполные или некорректные прогнозы FE в периоде {period}")
if not fe.period.isin(["tuning", "test"]).all():
    raise ValueError("Неожиданный период в FE_VALID")
control_score = day_score(fe.loc[fe.period.eq("test")], fe.loc[fe.period.eq("test"), "fe"])
if abs(control_score - 0.340950) > 0.00001:
    raise ValueError(f"Ожидался контроль FE 0,341: его локальная метрика 0,340950, получено {control_score:.6f}")
def compare_xgb():
    x = pd.read_parquet(XGB_VALID)
    joined = align(fe, x[KEY+["period","y","xgboost","model_info_through"]].rename(columns={"y":"x_y"}), KEY+["period"])
    if not np.isfinite(joined.xgboost).all() or not np.allclose(joined.y, joined.x_y) or not (joined.model_info_through < joined.cutoff).all():
        raise ValueError("Неверная цель или время доступности XGBoost")
    tune = joined.loc[joined.period.eq("tuning")]
    def mix(rows, w):
        return rows.fe * (1-w) + rows.xgboost * w
    weight = min(np.linspace(0,1,21), key=lambda w: day_score(tune, mix(tune,w)))
    rows = []
    for name, part in joined.groupby("period"):
        rows.append({"period":name,"FE":day_score(part,part.fe),"XGB":day_score(part,part.xgboost),
                     "mix":day_score(part,mix(part,weight)), "XGB weight":weight})
    display(pd.DataFrame(rows))
    if XGB_SUB.is_file():
        future = align(pd.read_csv(FE_SUB).rename(columns={"y_pred":"fe"}),
                       pd.read_csv(XGB_SUB).rename(columns={"y_pred":"xgboost"}), ["id"])
        if not np.isfinite(future[["xgboost", "fe"]]).all().all():
            raise ValueError("Пропущен прогноз в сабмите смеси")
        future["y_pred"] = mix(future,weight)
        future[["id","y_pred"]].sort_values("id").to_csv(OUT.parent/"submission_fe_xgb.csv",index=False)
if XGB_VALID.is_file():
    compare_xgb()
else:
    print("Для сравнения смеси укажите XGB_VALID.")

Для сравнения смеси укажите XGB_VALID.


## Данные и временные границы
Проверяется полная получасовая панель всех маршрутов. Границы задаются до расчёта любых статистик, использующих цель. Словарь корреляционных лагов обучается только до границы train.


In [7]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if not not history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 28:
    raise ValueError('Недостаточно дат для временного разбиения')
if len(dates) < 63:
    raise ValueError('Недостаточно истории для трёх недель подбора')
holdout_start = dates[-14]
folds = []
for start in (len(dates)-35, len(dates)-28, len(dates)-21):
    folds.append(dict(train=dates[:start-7], val=dates[start-7:start], tuning=dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))


,fold,period,first,last,days
0,1,train,2025-07-28 10:30:00,2025-09-19 10:30:00,54
1,1,val,2025-09-20 10:30:00,2025-09-26 10:30:00,7
2,1,tuning,2025-09-27 10:30:00,2025-10-03 10:30:00,7
3,1,test,2025-10-18 10:30:00,2025-10-31 10:30:00,14
4,2,train,2025-07-28 10:30:00,2025-09-26 10:30:00,61
5,2,val,2025-09-27 10:30:00,2025-10-03 10:30:00,7
6,2,tuning,2025-10-04 10:30:00,2025-10-10 10:30:00,7
7,2,test,2025-10-18 10:30:00,2025-10-31 10:30:00,14
8,3,train,2025-07-28 10:30:00,2025-10-03 10:30:00,68
9,3,val,2025-10-04 10:30:00,2025-10-10 10:30:00,7


## Исторические признаки и окна
Сохраняются признаки присланного ноутбука: уровни цели и статусов, средние за 2/6 часов, изменения за час, статусные суммы и отношения, суточные лаги, отклонение от среднего и отношение текущего уровня к предыдущим дням.

Вокруг каждого лага строится окно ±4 получасовых шага. Для последних значений и сглаженных сигналов доступны только точки не позже собственного старта: положительные сдвиги не создаются. Одинаковые источники не дублируются: например, соседние окна target_back_0/1/2/4 объединяются в target_back_0…8. Для status_latest добавляются back_1…4. Лаги сутки/двое назад и сезонные слоты получают все девять соседей. Исходные центральные признаки сохранены. Эквивалентные change_1h статусов 4–6 заменены на diff_hourly.

Суммы статусов не являются числом уникальных товаров. Названия warehouse_total_load, hub_buffer и hub_gate_bottleneck обозначают формулы, а не подтверждённые физические показатели. Недоступная история остаётся NaN.

Для цели и каждого статуса рассчитываются общая сумма по всем маршрутам в доступный момент и её изменение за час. Соседние исторические значения используют только левую часть окна у старта.


In [8]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

local_changes = {name: values[name] - past(values[name], positions - 2) for name in STATUSES[:3]}

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = local_changes[name]
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)


## Корреляционный лаг маршрута
На полной получасовой истории train ищется максимальная положительная Pearson-корреляция target(u) с diff_status6(u−L), L=2…48 шагов. Минимум 336 пар и корреляция >0,05 — эвристики, не тест значимости.

Выбранный L фиксирован для маршрута. Для горизонта h нужен сигнал origin+h−L. Помимо центра добавляются соседи ±2 часа, каждый доступен только при собственном источнике ≤origin. У каждого соседа есть флаг доступности. При слабой связи лаг и сигнал NaN. Для финального refit словарь пересчитывается на всей истории; оценочные parquet и модели сохраняются отдельно.

Для выбора лага и его применения из цели и изменения статуса 6 вычитается среднее того же получаса за предыдущие семь суток. Для каждого окна словарь строится заново до начала val.
Исходный лаг статуса 6 и лаг по остаткам сохраняются отдельными группами.


In [9]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

In [10]:
def horizon_dataset(h, origins, lag_table):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, lag_table)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

In [11]:
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
if get_gpu_device_count() < 1:
    raise RuntimeError('Для обучения включите GPU в настройках Kaggle Notebook.')
MODEL_DIR = OUT.parent / 'models'
MODEL_DIR.mkdir(exist_ok=True)
print('Начало подбора:', tune_start, 'Начало отложенной проверки:', holdout_start)

def make_model(trees, params):
    return CatBoostRegressor(iterations=trees, **params, loss_function='MAE', eval_metric='MAE', task_type='GPU', devices='0', cat_features=['route_id'], random_seed=42, allow_writing_files=False, verbose=200)

def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()
SEARCH_CONFIGS = [{'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'bootstrap_type': 'No'}, {'depth': 6, 'learning_rate': 0.07, 'l2_leaf_reg': 10, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1}, {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'bootstrap_type': 'No'}, {'depth': 8, 'learning_rate': 0.05, 'l2_leaf_reg': 10, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1}, {'depth': 10, 'learning_rate': 0.03, 'l2_leaf_reg': 10, 'bootstrap_type': 'No'}, {'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 3, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.5}]


from catboost import FeaturesData, Pool
FIXED = {2:(2,(2041,1285,2301)),6:(1,(1415,702,862))}

Начало подбора: 2025-10-11 10:30:00 Начало отложенной проверки: 2025-10-18 10:30:00


## Признаки и обучение
Пять стартов соответствуют 09:30–11:30. Время цели обучающей строки должно предшествовать началу val. Пиковые признаки используют по три соседних слота в каждом из семи предыдущих дней. Признаки рассчитываются порциями, затем один полный train каждого окна передаётся CatBoost. Для сопоставимости число деревьев в каждом окне берётся из прогона 0,341.

In [12]:
def pilot_frame(h, ix, lag_table, peaks):
    frame = horizon_dataset(h, ix, lag_table)
    phase = 2*np.pi*(times[ix].hour*2 + times[ix].minute//30)/48
    for name, value in [("sin", np.sin(phase)), ("cos", np.cos(phase))]:
        frame["start_"+name] = np.repeat(value,len(routes)).astype(np.float32)
    if peaks:
        windows = [np.stack([past(target, ix+h-48*d+k) for k in (-1,0,1)]) for d in range(1,8)]
        maxima = [np.max(w,axis=0) for w in windows]
        frame["peak7_mean_max"] = average_available(maxima).ravel()
        frame["peak7_mean_level"] = average_available([w.mean(axis=0) for w in windows]).ravel()
        for k in range(3):
            shares = [np.divide((w[k]==m).astype(float), (w==m).sum(axis=0),
                                out=np.full(m.shape,np.nan), where=np.isfinite(m)) for w,m in zip(windows,maxima)]
            frame[f"peak7_share_{k-1}"] = average_available(shares).ravel()
    return frame

def start_indices(h, count, boundary):
    ix = np.arange(len(times)-h)
    minutes = times[ix].hour*60+times[ix].minute
    if count == 1:
        ix = ix[minutes==630]
    elif count == 5:
        ix = ix[(minutes>=570)&(minutes<=690)]
    if boundary is not None:
        ix = ix[times[ix+h] < boundary]
    return ix

def build_pool(h, ix, lag_table, peaks, weight):
    first = pilot_frame(h,ix[:1],lag_table,peaks)
    cols = [c for c in first if c not in {"route_id","cutoff","timestamp","profile28","overlap"}]
    matrix = np.empty((len(ix)*len(routes),len(cols)), dtype=np.float32)
    for start in range(0,len(ix),16):
        batch = ix[start:start+16]
        matrix[start*len(routes):(start+len(batch))*len(routes)] = pilot_frame(h,batch,lag_table,peaks)[cols].to_numpy()
    route_codes = np.array([str(r).encode() for r in routes],dtype=object)
    features = FeaturesData(num_feature_data=matrix,cat_feature_data=np.tile(route_codes,len(ix)).reshape(-1,1),
                            num_feature_names=cols,cat_feature_names=["route_id"])
    mins = times[ix].hour*60+times[ix].minute
    weights = np.repeat(np.where((mins>=570)&(mins<=690),weight,1),len(routes))
    pool = Pool(features,label=target[ix+h].ravel()/TARGET_SCALE,weight=weights)
    return pool, cols+["route_id"]

lag_tables = [fit_lags(f["val"][0].normalize()) for f in folds]
def run(h,count,weight,peaks,fold_id,final=False):
    f = folds[fold_id]
    boundary = None if final else f["val"][0].normalize()
    lag_table = fit_lags(end+STEP) if final else lag_tables[fold_id]
    ix = start_indices(h,count,boundary)
    pool,cols = build_pool(h,ix,lag_table,peaks,weight)
    trial,trees = FIXED[h]
    model = make_model(trees[fold_id], SEARCH_CONFIGS[trial-1])
    model.fit(pool)
    train_pred = np.maximum(model.predict(pool)*TARGET_SCALE,0)
    train_meta = pd.DataFrame({"cutoff":times[ix].repeat(len(routes)),"y":target[ix+h].ravel()})
    train_row = score_row(train_meta,train_pred,"train")
    del pool, train_meta, train_pred
    gc.collect()
    records=[]
    diagnostics=[train_row]
    for name in (("future",) if final else ("val","tuning","test")):
        q = np.array([len(times)-1]) if final else times.get_indexer(f[name])
        part = pilot_frame(h,q,lag_table,peaks)
        pred = np.maximum(model.predict(part[cols])*TARGET_SCALE,0)
        meta = part[["cutoff","timestamp","route_id","profile28"]].copy()
        meta = meta.assign(horizon=h,period=name,catboost=pred)
        if not final:
            meta["y"] = target[q+h].ravel()
            diagnostics.append(score_row(meta,pred,name))
        records.append(meta)
    print(h,count,weight,peaks,fold_id+1,"final" if final else "")
    display(pd.DataFrame(diagnostics))
    if final:
        display(pd.read_csv(OUT.parent/f"report_{h}_{count}_{weight}_{peaks}_2.csv"))
    else:
        pd.DataFrame(diagnostics).to_csv(OUT.parent/f"report_{h}_{count}_{weight}_{peaks}_{fold_id}.csv",index=False)
    return pd.concat(records,ignore_index=True)

## Сравнение режимов
Сначала три числа стартов. Если пять стартов лучше 48 на tuning, проверяем веса 2 и 4 на полном наборе стартов. На h6 добавляем пиковые признаки к лучшему режиму. Выбираем по средней дневной метрике полной смеси на tuning, оставляя другие горизонты прогнозами версии 0,341. Отдельно смотрим WAPE пилота и результаты каждой недели. Последний test уже использовался в проекте.

In [13]:
def calibrated(frame, pred, h):
    raw = 0.7 * pred + 0.3 * frame.profile28
    return raw * (1.0264324341410613 if h == 2 else 1.0254124722109572)

def evaluate_variant(h,count,weight,peaks):
    panels = []
    for i in range(3):
        block = run(h,count,weight,peaks,i)
        mask = block.period.eq("tuning") | (block.period.eq("test") & (i==2))
        panels.append(block.loc[mask])
    panel = pd.concat(panels,ignore_index=True)
    panel["pilot"] = calibrated(panel,panel.catboost,h)
    if not np.isfinite(panel[["pilot", "catboost", "y"]]).all().all():
        raise ValueError("Некорректный прогноз пилота")
    paired = align(fe.loc[fe.horizon.eq(h)],panel[KEY+["period","pilot","catboost","y"]].rename(columns={"y":"pilot_y"}),KEY+["period"])
    if not np.array_equal(paired.y.to_numpy(), paired.pilot_y.to_numpy()):
        raise ValueError("Метки пилота отличаются от контрольных")
    combined = fe.merge(paired[KEY+["period","pilot"]],on=KEY+["period"],how="left",validate="one_to_one")
    combined["prediction"] = combined.pilot.fillna(combined.fe)
    result = {"h":h,"starts":count,"weight":weight,"peaks":peaks}
    for period,g in combined.groupby("period"):
        result[period] = day_score(g,g.prediction)
        result[period+"_FE"] = day_score(g,g.fe)
        sub = paired.loc[paired.period.eq(period)]
        result[period+"_WAPE_h"] = (sub.catboost-sub.y).abs().sum()/sub.y.sum()
    for i, fold in enumerate(folds, 1):
        weekly = combined.loc[combined.cutoff.isin(fold["tuning"])]
        result[f"week_{i}"] = day_score(weekly, weekly.prediction)
    panel.to_parquet(OUT.parent/f"pilot_{h}_{count}_{weight}_{peaks}.parquet",index=False)
    return result

experiments = []
chosen = {}
for h in PILOT:
    runs = [evaluate_variant(h,count,1,False) for count in (1,5,48)]
    if runs[1]["tuning"] < runs[2]["tuning"]:
        runs.extend(evaluate_variant(h,48,w,False) for w in (2,4))
    best = min(runs,key=lambda row:row["tuning"])
    if h == 6:
        runs.append(evaluate_variant(h,best["starts"],best["weight"],True))
    experiments.extend(runs)
    chosen[h] = min(runs,key=lambda row:row["tuning"])
    display(pd.DataFrame(runs))
pd.DataFrame(experiments).to_csv(OUT.parent/"pilot_metrics.csv",index=False)
joint = fe.copy()
joint["prediction"] = joint.fe
for h, best in chosen.items():
    if best["tuning"] < best["tuning_FE"]:
        file = OUT.parent / f"pilot_{h}_{best['starts']}_{best['weight']}_{best['peaks']}.parquet"
        indexed = pd.read_parquet(file).set_index(KEY+["period"]).pilot
        mask = joint.horizon.eq(h)
        replacement = indexed.reindex(pd.MultiIndex.from_frame(joint.loc[mask, KEY+["period"]]))
        if replacement.isna().any():
            raise ValueError("Пропущена замена горизонта")
        joint.loc[mask,"prediction"] = replacement.to_numpy()
joint_report = pd.DataFrame([{"period":name,"FE":day_score(rows, rows.fe),
                              "pilot":day_score(rows, rows.prediction)} for name,rows in joint.groupby("period")])
joint_report.to_csv(OUT.parent/"joint_metrics.csv",index=False)
display(joint_report)
use_pilot = bool(joint_report.loc[joint_report.period.eq("tuning"), "pilot"].item() <
                 joint_report.loc[joint_report.period.eq("tuning"), "FE"].item())


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2390480	total: 123ms	remaining: 4m 10s
200:	learn: 0.8259257	total: 2.95s	remaining: 27s
400:	learn: 0.8058664	total: 5.62s	remaining: 23s
600:	learn: 0.7915505	total: 8.46s	remaining: 20.3s
800:	learn: 0.7788364	total: 11.3s	remaining: 17.5s
1000:	learn: 0.7672688	total: 14.1s	remaining: 14.7s
1200:	learn: 0.7563121	total: 16.9s	remaining: 11.8s
1400:	learn: 0.7456680	total: 19.7s	remaining: 9.02s
1600:	learn: 0.7357226	total: 22.6s	remaining: 6.2s
1800:	learn: 0.7260203	total: 25.3s	remaining: 3.37s
2000:	learn: 0.7166387	total: 28.1s	remaining: 562ms
2040:	learn: 0.7149413	total: 28.6s	remaining: 0us
2 1 1 False 1 


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,54,54000,0.274640,-0.034735,0.034735,0.309375,0.309898
1,val,7,7000,0.319437,-0.046207,0.046207,0.365644,0.369906
2,tuning,7,7000,0.309987,-0.038970,0.038970,0.348957,0.349218
3,test,14,14000,0.312583,-0.072399,0.072399,0.384982,0.384613


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2582884	total: 23.4ms	remaining: 30.1s
200:	learn: 0.8402881	total: 2.92s	remaining: 15.7s
400:	learn: 0.8211234	total: 5.7s	remaining: 12.6s
600:	learn: 0.8079394	total: 8.5s	remaining: 9.67s
800:	learn: 0.7964351	total: 11.3s	remaining: 6.82s
1000:	learn: 0.7859092	total: 14s	remaining: 3.98s
1200:	learn: 0.7760494	total: 16.8s	remaining: 1.18s
1284:	learn: 0.7720517	total: 18s	remaining: 0us
2 1 1 False 2 


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,61,61000,0.292300,-0.036235,0.036235,0.328535,0.328915
1,val,7,7000,0.309647,-0.034075,0.034075,0.343722,0.345112
2,tuning,7,7000,0.315589,-0.029853,0.029853,0.345442,0.344958
3,test,14,14000,0.312155,-0.070238,0.070238,0.382393,0.381978


... (74 outputs left out)

0:	learn: 1.2741324	total: 48.5ms	remaining: 41.8s
200:	learn: 0.9248790	total: 5.55s	remaining: 18.3s
400:	learn: 0.9025628	total: 11s	remaining: 12.6s
600:	learn: 0.8983095	total: 16.2s	remaining: 7.04s
800:	learn: 0.8958385	total: 21.5s	remaining: 1.64s
861:	learn: 0.8951691	total: 23.2s	remaining: 0us
6 5 1 True 3 


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,68,340000,0.339832,-0.039804,0.039804,0.379636,0.381546
1,val,7,7000,0.341560,-0.032334,0.032334,0.373894,0.379364
2,tuning,7,7000,0.347014,-0.039928,0.039928,0.386942,0.400067
3,test,14,14000,0.347103,-0.044443,0.044443,0.391546,0.393898


,h,starts,weight,peaks,test,test_FE,test_WAPE_h,tuning,tuning_FE,tuning_WAPE_h,week_1,week_2,week_3
0,6,1,1,False,0.340758,0.34095,0.348488,0.339518,0.339504,0.342781,0.332904,0.343747,0.341902
1,6,5,1,False,0.340953,0.34095,0.347219,0.339082,0.339504,0.342519,0.332623,0.343512,0.341112
2,6,48,1,False,0.340369,0.34095,0.348641,0.339197,0.339504,0.344160,0.332671,0.344107,0.340814
3,6,48,2,False,0.340390,0.34095,0.348351,0.339157,0.339504,0.343894,0.332631,0.344059,0.340780
4,6,48,4,False,0.340477,0.34095,0.347875,0.339136,0.339504,0.343494,0.332669,0.343880,0.340860
5,6,5,1,True,0.340821,0.34095,0.347103,0.338980,0.339504,0.342464,0.332437,0.343504,0.341001


,period,FE,pilot
0,test,0.340950,0.339532
1,tuning,0.339504,0.339124


## Сабмит
Все выигравшие пилотные модели переобучаются на всей доступной размеченной истории своего режима. Для других горизонтов используются предсказания исходного финального FE. Исходный FE остаётся кандидатом, если пилот не улучшил tuning. Независимая оценка test получена до финального переобучения.

In [14]:
future = fe_future.copy()
for h, best in chosen.items():
    if not use_pilot or best["tuning"] >= best["tuning_FE"]:
        continue
    panel = run(h,best["starts"],best["weight"],best["peaks"],2,True)
    pred = panel.assign(update=calibrated(panel,panel.catboost,h))[["route_id","timestamp","update"]]
    mask = future.timestamp.eq(end+h*STEP)
    updated = future.loc[mask].merge(pred,on=["route_id","timestamp"],how="left",validate="one_to_one")
    if not np.isfinite(updated["update"]).all():
        raise ValueError("Неполный прогноз")
    future.loc[mask,"y_pred"] = updated["update"].to_numpy()
future[["id","y_pred"]].to_csv(OUT.parent/"submission_pilot.csv",index=False)

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2680063	total: 465ms	remaining: 17m 48s
200:	learn: 0.8585217	total: 1m 12s	remaining: 12m 35s
400:	learn: 0.8488396	total: 2m 22s	remaining: 11m 15s
600:	learn: 0.8439182	total: 3m 32s	remaining: 10m 2s
800:	learn: 0.8406336	total: 4m 41s	remaining: 8m 47s
1000:	learn: 0.8381289	total: 5m 50s	remaining: 7m 35s
1200:	learn: 0.8360513	total: 6m 57s	remaining: 6m 22s
1400:	learn: 0.8344061	total: 8m 4s	remaining: 5m 11s
1600:	learn: 0.8329223	total: 9m 11s	remaining: 4m 1s
1800:	learn: 0.8316544	total: 10m 18s	remaining: 2m 51s
2000:	learn: 0.8304935	total: 11m 26s	remaining: 1m 42s
2200:	learn: 0.8295096	total: 12m 34s	remaining: 34.3s
2300:	learn: 0.8290710	total: 13m 9s	remaining: 0us
2 48 1 False 3 final


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,97,4628000,0.317321,-0.034559,0.034559,0.35188,0.352133


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,68,3262000,0.316344,-0.033373,0.033373,0.349717,0.350188
1,val,7,7000,0.314067,-0.027293,0.027293,0.341360,0.350654
2,tuning,7,7000,0.306467,-0.028350,0.028350,0.334817,0.335146
3,test,14,14000,0.307841,-0.053335,0.053335,0.361176,0.360827


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.3064796	total: 61.1ms	remaining: 52.6s
200:	learn: 0.9487758	total: 7.2s	remaining: 23.7s
400:	learn: 0.9261732	total: 14.3s	remaining: 16.5s
600:	learn: 0.9219393	total: 21.1s	remaining: 9.16s
800:	learn: 0.9197469	total: 27.8s	remaining: 2.12s
861:	learn: 0.9191924	total: 29.9s	remaining: 0us
6 5 1 True 3 final


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,96,480000,0.339608,-0.040714,0.040714,0.380323,0.381604


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,68,340000,0.339832,-0.039804,0.039804,0.379636,0.381546
1,val,7,7000,0.341560,-0.032334,0.032334,0.373894,0.379364
2,tuning,7,7000,0.347014,-0.039928,0.039928,0.386942,0.400067
3,test,14,14000,0.347103,-0.044443,0.044443,0.391546,0.393898
